# Signed Distance Field

Mapping sampling points of UV space to nearest points on a curve.

Assuming the curve is almost flat with some little bumps/dents. Dropping $z$ for all calculations.

- $p$ — sample point in UV space
- $p^* = S(t^*)$ — nearest point on curve

$$
p|_{S(t)} \to p^* = S(t^*), t^* = \argmin_t{||p - S(t)||^2}
$$

### General solution

- $S(t)$ — general parametric curve
- $S'(t)$ ­— first derivative, velocity/flow

$$
(p - S(t))·S'(t) = 0
\tag{perpendicularity}
$$

Solving for $t$ gives points with locally minimum distance.
Globally minimum should be choosen somehow by additional computations.

### Distance

- $v_p = p - p^*$ — vector from a projection point to the sample

$$D^2 = ||v_p||^2 = v_p·v_p$$

Doesn't make sense to squaeroot because it can be used as is.

### Orientation

- $v = p - p^*$ — vector from a projection point to the sample
- $f = S'(t^*)$ ­— tangential flow at the projection point
- $z = (v × f)_z = v_x f_y - v_y f_x$ — z-component of cross product of 2D vectors into 3D

$$
\begin{cases}
z > 0 \implies \text{at right-hand side} \\
z < 0 \implies \text{at left-hand side} \\
z = 0 \implies \text{exact on the curve, should not happen}\\
\end{cases}
$$

Numeric value $z = |v||f|\sin\phi$


In [124]:
from typing import NamedTuple

import numpy as np
from numpy.typing import NDArray

Vec = np.ndarray[tuple[int], np.dtype[np.floating]]  # 2D or 3D
vec = lambda *args: np.array(args, dtype=float)
nparr = lambda *args: np.array(args)
npgen = lambda iter: np.fromiter(iter, dtype=float)


dot = lambda v1, v2: float(v1 @ v2)
dots = lambda a1, a2: (a1 * a2).sum(axis=-1)

In [125]:
def powers(d: int, t: float) -> NDArray:
    return np.power(t, np.arange(d + 1))


def powersnp(d: int, t: NDArray) -> NDArray:
    return np.power(t[..., None], np.arange(d + 1))

In [126]:
def random_points(num: int, jigg=0.25):
    """Geneates control points
    first and ast fized at corners (1, 0) and (0, 1),
    midpoints jiggled
    """
    xx = np.linspace(1.0, 0.0, num)
    yy = np.linspace(0.0, 1.0, num)

    coords = np.stack((xx, yy)).T
    coords[1:-1] += np.random.randn(num - 2, 2) * jigg
    return coords

In [127]:
def beziefy_controls(p1: NDArray, p2: NDArray, p3: NDArray) -> tuple[NDArray, NDArray, NDArray]:
    """Bezier control points from bspline segment points"""
    return (p1 + p2) * 0.5, p2, (p2 + p3) * 0.5

In [128]:
def disquance(p0: Vec, p1: Vec):
    v = p1 - p0
    return dot(v, v)


def disquances(p0: Vec, pp: NDArray):
    vv = pp - p0
    return (vv * vv).sum(axis=1)


def orientation(frw: Vec, vec: Vec):
    return np.sign(vec[0] * frw[1] - vec[1] * frw[0])


def orientations(frws: NDArray, vecs: NDArray):
    return np.sign(vecs[:, 0] * frws[:, 1] - vecs[:, 1] * frws[:, 0])

In [129]:
def gridspace(lowbound: float, highbound: float, res: int) -> NDArray:
    """Generate XY space between bounds
    with coordinates in centers of grid cells
    """
    r = (highbound - lowbound) / res / 2  # pixel half-size
    cspace = np.linspace(lowbound, highbound, res, endpoint=False) + r
    X, Y = np.meshgrid(cspace, cspace)
    return np.stack((X, Y), axis=-1)

In [130]:
from proj import Projections, saveproj

#### SDF View widget


In [131]:
from collections.abc import Callable
from traitlets import Instance, Tuple, observe
from ipywidgets import widgets as wg
from plotly import graph_objects as go
from plotly import colors


class SDFView(wg.Box):
    controls = Instance[NDArray | None](np.ndarray, allow_none=True)
    "controls a plotted as polygon"

    params = Tuple(allow_none=True)
    "params are passed to functions directly"

    plotting: Callable
    mapping: Callable
    picking: Callable

    tspace_curve = Instance[NDArray](np.ndarray, default_value=np.linspace(0.0, 1.0, 16))
    gspace_field = Instance(np.ndarray, default_value=gridspace(0.0, 1.0, 32))

    figure: go.FigureWidget
    out: wg.Output

    def __init__(self, plotting, mapping, picking, **kwargs):
        self.plotting = plotting
        self.mapping = mapping
        self.picking = picking

        self.figure = go.FigureWidget(
            layout=go.Layout(
                autosize=False,
                height=720,
                margin=dict(t=16, b=16, l=16, r=16),
                xaxis=dict(scaleanchor="y"),
                yaxis=dict(range=(0, 1)),
            ),
            data=[
                go.Heatmap(
                    name="field",
                    x=[0.5],
                    y=[0.5],
                    z=[0.0],
                    zmid=0.0,
                    colorscale=colors.diverging.RdBu_r,
                    showscale=False,
                    showlegend=True,
                    hovertemplate="xy: %{x:.03f} %{y:.03f}<br>dist: %{z:.03f}",
                ),
                go.Scatter(
                    name="curve",
                    x=[0, 0],
                    y=[0, 0],
                    mode="lines",
                    hovertemplate="t: %{customdata:.03f}<br>xy: %{x:.03f} %{y:.03f}",
                    line=dict(width=4, color="black"),
                    showlegend=True,
                ),
                go.Scatter(
                    name="controls",
                    mode="markers+lines",
                    marker=dict(symbol="circle-open", color="black"),
                    opacity=0.25,
                    hoverinfo="skip",
                ),
                go.Scatter(
                    name="pick",
                    x=[],
                    y=[],
                    text=[],
                    mode="markers+text",
                    marker=dict(color="purple"),
                    textposition="top center",
                    hoverinfo="skip",
                    showlegend=False,
                ),
            ],
        )

        self.out = wg.Output()

        super().__init__(children=[self.figure], **kwargs)

        self.figure.data[0].on_click(self.on_pick)  # type: ignore

    @observe("controls")
    def update_controls(self, _):
        selector = {"name": "controls"}
        if self.controls is None:
            self.figure.update_traces(dict(x=[], y=[]), selector)
            return

        points = self.controls.T
        self.figure.update_traces(dict(x=points[0], y=points[1]), selector)
        self.reset_pick()

    @observe("params", "tspace_curve")
    def update_curve(self, _):
        selector = {"name": "curve"}
        if self.params is None:
            self.figure.update_traces(dict(x=[], y=[], customdata=[]), selector)
            return

        curve = self.plotting(self.tspace_curve, self.params).T

        self.figure.update_traces(
            dict(
                x=curve[0],
                y=curve[1],
                customdata=self.tspace_curve,
            ),
            selector,
        )

    @observe("params", "gspace_field")
    def update_field(self, _):
        selector = {"name": "field"}

        if self.controls is None:
            self.figure.update_traces(dict(x=[], y=[], z=[], customdata=[]), selector)
            return

        lattice = self.gspace_field.reshape((self.gspace_field.size // 2, 2))
        projs = self.mapping(lattice, self.params)
        distfield = projs.sdist

        self.figure.update_traces(
            dict(
                x=lattice[:, 0],
                y=lattice[:, 1],
                z=distfield,
                customdata=[projs.arc],
            ),
            selector,
        )

    def on_pick(self, trace, points, mouse):
        if self.controls is None or len(points.point_inds) == 0:
            return

        assert len(points.point_inds) == 1

        self.update_pick(nparr(points.xs[0], points.ys[0]))

    def update_pick(self, sample: Vec):
        projs = self.picking(sample, self.params)
        markers = projs.pnt
        labels = [f"t: {t:.03f}<br>dsq: {d:.03f}" for t, d in zip(projs.t, projs.dsq)]
        markers = np.array(markers).T

        self.figure.update_traces(
            dict(
                x=markers[0],
                y=markers[1],
                text=labels,
            ),
            {"name": "pick"},
        )

    def reset_pick(self):
        self.figure.update_traces(dict(x=[], y=[], text=[]), {"name": "pick"})


# Quadratic Beziel

Bezier segment over $c_1, c_2, c_3$ localized around its apex with $t=0.0$.


- $v_1 = c_1 - c_2$
- $v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axial vector
- $v_t = v_3 - v_1$ — parabolic tangential vector
- $c_a = c_2 + .25 v_a = S(0)$ — apex point

$$
S(t') - c_a=
\big[1, t', t'^2 \big]
\begin{bmatrix}
0 \\
v_t \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t', t'^2 \big]
\begin{bmatrix}
v_t \\
2 v_a \\
0 \\
\end{bmatrix}
$$


### Distance

- $p_a = p - c_a$ — point relative to apex
- $v_p(t)|p = p - S(t)$ ­— vector from curve to sample point
- $v_p^2 = v_p·v_p = |v_p|^2$ — squared distance from curve to point

Equation for local minima/maxima:

$$
\frac{dv_p^2}{dt} =
[1, t, t^2, t^3]
\begin{pmatrix}
-p_a v_t \\
-2 p_a v_a + v_t^2 \\
3 v_a v_t \\
2 v_a^2 \\
\end{pmatrix}
= 0
$$

There are 1 up to 3 bending points. Need to select the best.


In [132]:
def beziel_local(c1: Vec | NDArray, c2: Vec | NDArray, c3: Vec | NDArray):
    """Calculate local basis c2 v1 c3"""
    return c2, c1 - c2, c3 - c2


def beziel_parab(c1: Vec | NDArray, c2: Vec | NDArray, c3: Vec | NDArray):
    """Calculate parabolic basis ca vt va"""
    v1 = c1 - c2
    v3 = c3 - c2
    va = v3 + v1
    vt = v3 - v1
    ca = c2 + 0.25 * va
    return ca, vt, va


def beziel_M(ca: Vec, vt: Vec, va: Vec):
    return np.array([np.zeros(ca.shape), vt, va])


def beziel_Mdt(ca: Vec, vt: Vec, va: Vec):
    return np.array([vt, 2 * va, np.zeros(ca.shape)])


def beziel_curve(tspace: NDArray, ca: Vec, vt: Vec, va: Vec):
    MC = beziel_M(ca, vt, va)
    return powersnp(2, tspace - 0.5) @ MC + ca


def beziel_curve1(t: float, ca: Vec, vt: Vec, va: Vec):
    MC = beziel_M(ca, vt, va)
    return powers(2, t - 0.5) @ MC + ca


def beziel_flow(tspace: NDArray, ca: Vec, vt: Vec, va: Vec):
    MC = beziel_Mdt(ca, vt, va)
    return powersnp(2, tspace - 0.5) @ MC


def beziel_flow1(t: float, ca: Vec, vt: Vec, va: Vec):
    MC = beziel_Mdt(ca, vt, va)
    return powers(2, t - 0.5) @ MC

In [133]:
def equation_np(ca: Vec, vt: Vec, va: Vec, sample: Vec) -> NDArray:
    """Solve the equation of perpendicularity = square distance local extrema
    a0 + a1 t + a2 t^2 + a3 t^3
    """
    p = ca - sample  # -p_a
    dot_aa = dot(va, va)
    dot_at = dot(va, vt)
    dot_tt = dot(vt, vt)
    dot_pa = dot(p, va)
    dot_pt = dot(p, vt)

    a0 = dot_pt
    a1 = 2 * dot_pa + dot_tt
    a2 = 3 * dot_at
    a3 = 2 * dot_aa

    roots = np.roots([a3, a2, a1, a0])
    roots_real = roots[roots.imag == 0].real
    roots_uniq = npgen(set(roots_real))

    return roots_uniq

In [134]:
solve_equ = equation_np

In [135]:
def projections_all(sample: Vec, ca: Vec, vt: Vec, va: Vec) -> Projections:
    tt = solve_equ(ca, vt, va, sample) + 0.5
    pnts = beziel_curve(tt, ca, vt, va)
    vecs = sample - pnts
    dsqs = dots(vecs, vecs)
    flws = beziel_flow(tt, ca, vt, va)
    cnt = tt.shape[0]
    return Projections(
        np.tile(sample, (cnt, 1)),
        tt,
        pnts,
        dsqs,
        flws,
        orientations(vecs, flws),
    )


def projection_best(sample: Vec, ca: Vec, vt: Vec, va: Vec) -> Projections:
    tt = solve_equ(ca, vt, va, sample) + 0.5

    pnts = beziel_curve(tt, ca, vt, va)
    vecs = sample - pnts
    dsqs = dots(vecs, vecs)

    best = np.argmin(dsqs)
    t = tt[best]
    pnt = pnts[best]
    vec = vecs[best]
    dsq = dsqs[best]
    flw = beziel_flow1(t, ca, vt, va)
    sgn = orientation(flw, vec)

    return Projections(
        nparr(sample),
        nparr(t),
        nparr(pnt),
        nparr(dsq),
        nparr(flw),
        nparr(sgn),
    )

In [136]:
def projection_field(samples: NDArray, ca: Vec, vt: Vec, va: Vec) -> Projections:
    projs = [projection_best(s, ca, vt, va) for s in samples]
    return Projections(
        samples,
        np.concat([p.arc for p in projs]),
        np.concat([p.pnt for p in projs]),
        np.concat([p.dsq for p in projs]),
        np.concat([p.flw for p in projs]),
        np.concat([p.sgn for p in projs]),
    )

In [ ]:
view = SDFView(
    plotting=lambda tspace, params: beziel_curve(tspace, *params),
    mapping=lambda samples, params: projection_field(samples, *params),
    picking=lambda sample, params: projections_all(sample, *params),
)

display(view)

In [137]:
controls = random_points(3)
p1, p2, p3 = controls
c1, c2, c3 = beziefy_controls(p1, p2, p3)

view.controls = controls
view.params = beziel_parab(c1, c2, c3)

---


In [138]:
controls = random_points(3)
p1, p2, p3 = controls
c1, c2, c3 = beziefy_controls(p1, p2, p3)
params = beziel_parab(c1, c2, c3)

In [139]:
RES = 16
lattice = gridspace(0.0, 1.0, RES).reshape((RES * RES, 2))
field = projection_field(lattice, *params)

In [140]:
saveproj(field, "test.npz")